# Task4 — Train-only quality-conditioned ensemble

Run All ฝึก HGB router ขนาดเล็กบน Train แล้วใช้ prediction cache ของ CNN เดิม ไม่มีการฝึกภาพใหม่ ไม่เปิด Test และไม่สร้าง submission. น้ำหนัก compound = 0.5 + 0.5 × router probability; sampler = ส่วนที่เหลือ. สูตรน้ำหนัก fixed ก่อนประเมิน ไม่จูนตาม OOF errors.

Router เรียนจำแนก augmentation branch (original/mild vs added compound) ไม่ใช่ความถูกต้องของ expert และ original อาจมี corruption อยู่แล้ว. จึงเป็น proxy ที่อาจผิดหรือทำให้คะแนนลดได้. คงภาพ raw ก่อน padding สำหรับการวัดคุณภาพ.

Router ของแต่ละfoldฝึกเฉพาะ training rows ของ foldนั้น ใช้ seedคงที่100ภาพ/class. Validationใช้held-out rowsเท่านั้น. แต่โมเดลฐาน/protocolเคยเลือกจาก OOF นี้แล้ว จึงยังเป็น exploratory validation ไม่ใช่ independent performance guarantee. ไม่ใช้ audit รอบเก่ามาปรับสูตร.

## 1. Load frozen configuration and predictions

In [1]:
from pathlib import Path
import sys,json,hashlib,gc
import numpy as np
import pandas as pd
import torch,joblib
from threadpoolctl import threadpool_limits
CPU_LIMIT=threadpool_limits(limits=4)
from PIL import Image
from sklearn.metrics import f1_score
from IPython.display import display
ROOT=next(p for b in (Path.cwd(),*Path.cwd().parents) for p in (b,b/'CPE342-Kaggle') if (p/'Task4/models/image_quality_router.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from Task4.models import image_quality_router as routing
from Task4.models.image_followup import probabilities
from Task4.models.image_compound_transforms import CONDITIONS,corrupt
RAW=ROOT/'Task4/data/raw'
SOURCE=ROOT/'Task4/models/results/compound_supervised_v1'
HEAVY=ROOT/'Task4/models/results/heavy_balance_v1/heavy_sampler18'
OUT=ROOT/'Task4/models/results/quality_router_v1'
train=pd.read_csv(RAW/'train.csv');assert train.id.is_unique and train.label.notna().all()
torch.set_num_threads(4)
sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()
config={'policy':routing.POLICY,'router_sha':sha(Path(routing.__file__)),'train_sha':sha(RAW/'train.csv'),
        'source_settings_sha':sha(SOURCE/'settings.json'),'condition_source_sha':sha(ROOT/'Task4/models/image_compound_transforms.py'),
        'reference_protocol':'fixed 50/50 and 75/25; no Test tuning'}
OUT.mkdir(parents=True,exist_ok=True)
p=OUT/'config.json'
if p.exists():assert json.loads(p.read_text())==config,'Changed config: choose a new OUT'
else:p.write_text(json.dumps(config,indent=2))


## 2. Fit each router on matching Train folds and evaluate held-out original images

In [2]:
records=[];routers={};fold_frames={}
def score(y,p):return f1_score(y,p.argmax(1),labels=range(5),average='macro',zero_division=0)
def evaluate(fold,suite,condition,y,a,b,x):
    q=routers[fold].predict_proba(x)[:,1];r,w=routing.blend(a,b,q)
    for name,p in [('fixed50',.5*a+.5*b),('fixed75',.75*a+.25*b),('router',r)]:
        records.append({'fold':fold,'suite':suite,'condition':condition,'procedure':name,'F1':score(y,p),'rows':len(y)})
    return q,w
for fold in range(1,6):
    a=pd.read_parquet(SOURCE/'compound_supervised6'/f'fold{fold}'/'oof.parquet')
    b=pd.read_parquet(HEAVY/f'fold{fold}'/'oof.parquet')
    assert a.id.tolist()==b.id.tolist() and np.array_equal(a.true,b.true)
    valid=train.set_index('id').loc[a.id].reset_index()
    assert np.array_equal(valid.label,a.true)
    fit=train.loc[~train.id.isin(valid.id)]
    assert set(fit.id).isdisjoint(valid.id)
    d=OUT/f'fold{fold}';d.mkdir(exist_ok=True)
    signature={'fit_ids_sha':hashlib.sha256('\n'.join(fit.id).encode()).hexdigest(),
        'oof_sha':[sha(SOURCE/'compound_supervised6'/f'fold{fold}'/'oof.parquet'),sha(HEAVY/f'fold{fold}'/'oof.parquet')],'config':config}
    sig=d/'signature.json'
    if sig.exists():assert json.loads(sig.read_text())==signature
    else:sig.write_text(json.dumps(signature,indent=2))
    if (d/'router.joblib').exists():router=joblib.load(d/'router.joblib')
    else:
        print('Fit router',fold,flush=True)
        router,ids=routing.fit_router(RAW,fit)
        assert set(ids).isdisjoint(valid.id)
        (d/'training_ids.json').write_text(json.dumps(ids))
        temp=d/'router.partial.joblib';joblib.dump(router,temp);temp.replace(d/'router.joblib')
    routers[fold]=router;fold_frames[fold]=(a,b)
    fp=d/'original_features.npy'
    if fp.exists():x=np.load(fp)
    else:
        feats=[]
        for row in valid.itertuples():
            with Image.open(RAW/'train'/row.file_name) as f:feats.append(routing.features(f.convert('RGB')))
        x=np.stack(feats);np.save(fp,x)
    assert len(x)==len(valid)
    q,w=evaluate(fold,'original','original',a.true.to_numpy(int),probabilities(a,'tta'),probabilities(b,'tta'),x)
    pd.DataFrame({'id':valid.id,'router_probability':q,'compound_weight':w}).to_parquet(d/'original_routing.parquet',index=False)
    print('Fold',fold,'mean compound weight',round(float(w.mean()),4),flush=True)
display(pd.DataFrame(records).pivot(index='fold',columns='procedure',values='F1'))


Fit router 1
Fold 1 mean compound weight 0.6406
Fit router 2
Fold 2 mean compound weight 0.6398
Fit router 3
Fold 3 mean compound weight 0.6357
Fit router 4
Fold 4 mean compound weight 0.6309
Fit router 5
Fold 5 mean compound weight 0.6427


procedure,fixed50,fixed75,router
fold,,,
1,0.981367,0.979212,0.979890
2,0.980392,0.978720,0.979440
3,0.982930,0.981066,0.981362
4,0.978785,0.976841,0.977401
5,0.980762,0.978916,0.980209


## 3. Check development corruption suites using the same held-fold router

In [3]:
for fold in range(1,6):
    sample=pd.read_parquet(SOURCE/f'stress_sample_fold{fold}.parquet')
    assert set(sample.id).issubset(set(fold_frames[fold][0].id))
    for condition,(phase,_,_) in CONDITIONS.items():
        if phase!='development':continue
        frames=[pd.read_parquet(SOURCE/'stress_cache'/f'fold{fold}'/m/f'{condition}.parquet') for m in ['candidate','sampler']]
        assert all(f.id.tolist()==sample.id.tolist() for f in frames)
        a,b=[f[[f'p{i}' for i in range(5)]].to_numpy() for f in frames]
        fp=OUT/f'fold{fold}'/f'{condition}_features.npy'
        if fp.exists():x=np.load(fp)
        else:
            feats=[]
            for row in sample.itertuples():
                with Image.open(RAW/'train'/row.file_name) as f:img=f.convert('RGB')
                feats.append(routing.features(corrupt(img,condition,row.id)))
            x=np.stack(feats);np.save(fp,x)
        q,w=evaluate(fold,'compound' if condition.startswith('compound_') else 'single',condition,sample.label.to_numpy(int),a,b,x)
    print('Stress completed fold',fold,flush=True)
results=pd.DataFrame(records)
results.to_parquet(OUT/'evaluation.parquet',index=False)
summary=results.groupby(['suite','procedure']).F1.mean().unstack()
display(summary)
display(summary.subtract(summary.fixed50,axis=0))


Stress completed fold 1
Stress completed fold 2
Stress completed fold 3
Stress completed fold 4
Stress completed fold 5


procedure,fixed50,fixed75,router
suite,,,
compound,0.926588,0.934865,0.935083
original,0.980847,0.978951,0.979660
single,0.975847,0.974312,0.974997


procedure,fixed50,fixed75,router
suite,,,
compound,0.0,0.008277,0.008495
original,0.0,-0.001896,-0.001187
single,0.0,-0.001535,-0.000850


## 4. Evidence only — no automatic Test inference

In [4]:
fold_summary=results.groupby(['fold','suite','procedure']).F1.mean().unstack()
delta=fold_summary.router-fold_summary.fixed50
display(delta.unstack('suite'))
report={'mean_gain_vs_fixed50':{k:float(v) for k,v in delta.groupby('suite').mean().items()},
        'fold_wins':{k:int(v) for k,v in (delta>0).groupby('suite').sum().items()},
        'test_used':False,'auto_export':False,'note':'Exploratory evaluation on reused OOF; inspect tradeoffs before fitting a final router or exporting'}
(OUT/'results.json').write_text(json.dumps(report,indent=2))
display(report)
print('Saved:',OUT,'No full router or Test predictions generated. ไม่ปรับน้ำหนักตามผลโดยอัตโนมัติ')


suite,compound,original,single
fold,,,
1,0.009025,-0.001477,0.000370
2,0.003927,-0.000952,-0.003011
3,0.012343,-0.001568,-0.002008
4,0.006393,-0.001384,0.001208
5,0.010790,-0.000552,-0.000811


{'mean_gain_vs_fixed50': {'compound': 0.008495473913205953,
  'original': -0.0011867607473780907,
  'single': -0.0008504636624923068},
 'fold_wins': {'compound': 5, 'original': 0, 'single': 2},
 'test_used': False,
 'auto_export': False,
 'note': 'Exploratory evaluation on reused OOF; inspect tradeoffs before fitting a final router or exporting'}

Saved: /Users/meng/dev/MyUniversity/CPE342-MachineLearning/CPE342-Kaggle/Task4/models/results/quality_router_v1 No full router or Test predictions generated. ไม่ปรับน้ำหนักตามผลโดยอัตโนมัติ
